# Machine Clustering based on Mean Metrics

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

### Load Data and Group by MachineID

In [3]:
df = pd.read_excel('QUALITY & DEFECT REDUCTION.xlsx', engine='openpyxl')

# Calculate mean metrics per MachineID
machine_stats = df.groupby('MachineID')[['DefectRate', 'Vibration', 'Temperature', 'Pressure']].mean().reset_index()
machine_stats.head()

BadZipFile: File is not a zip file

### Perform KMeans Clustering

In [ ]:
# Define features for clustering
X = machine_stats[['DefectRate', 'Vibration', 'Temperature', 'Pressure']]

# Initialize and fit KMeans
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
machine_stats['Cluster'] = kmeans.fit_predict(X)


### Label Clusters based on Centers

In [ ]:
# Calculate global means to compare against cluster centers
global_means = X.mean()

cluster_labels = {}
for i, center in enumerate(kmeans.cluster_centers_):
    # center represents mean of [DefectRate, Vibration, Temperature, Pressure] for the cluster
    defect = "High Defect" if center[0] > global_means['DefectRate'] else "Low Defect"
    vibration = "High Vibration" if center[1] > global_means['Vibration'] else "Low Vibration"
    temperature = "High Temp" if center[2] > global_means['Temperature'] else "Low Temp"
    pressure = "High Pressure" if center[3] > global_means['Pressure'] else "Low Pressure"
    
    # Creating a combined label
    label = f"{defect} / {vibration} / {temperature} / {pressure}"
    cluster_labels[i] = label

machine_stats['Cluster_Label'] = machine_stats['Cluster'].map(cluster_labels)


### Output Cluster Summary

In [ ]:
# Summarize the number of machines in each cluster
summary = machine_stats.groupby(['Cluster', 'Cluster_Label']).size().reset_index(name='Machine_Count')
print("Cluster Summary:")
display(summary)

print("\nSample of Clustered Machines:")
display(machine_stats.head())

### Visualize Clusters

In [ ]:
plt.figure(figsize=(15, 6))

# Scatter plot: DefectRate vs Vibration
plt.subplot(1, 2, 1)
sns.scatterplot(data=machine_stats, x='DefectRate', y='Vibration', hue='Cluster_Label', palette='viridis', s=100)
plt.title('Defect Rate vs Vibration')
plt.xlabel('Mean Defect Rate')
plt.ylabel('Mean Vibration')
plt.legend(bbox_to_anchor=(0, -0.15), loc='upper left', ncol=1)

# PCA 2D Projection
pca = PCA(n_components=2)
components = pca.fit_transform(X)
machine_stats['PCA1'] = components[:, 0]
machine_stats['PCA2'] = components[:, 1]

plt.subplot(1, 2, 2)
sns.scatterplot(data=machine_stats, x='PCA1', y='PCA2', hue='Cluster_Label', palette='viridis', s=100)
plt.title('Clusters in 2D PCA Space')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(bbox_to_anchor=(0, -0.15), loc='upper left', ncol=1)

plt.tight_layout()
plt.show()